# Notebook 01: ML System Design Framework

## Why This Matters

ML system design interviews separate engineers who can implement models from engineers who can build production systems. Top-tier companies (Google, Meta, Airbnb, Netflix) use ML system design as a key signal for senior roles because real systems fail not because of model accuracy, but because of wrong problem framing, data pipeline bugs, or misaligned metrics. A structured framework lets you think clearly under pressure and communicate your decisions to interviewers, teammates, and stakeholders. The difference between a junior and senior ML engineer is often this: seniors ask 'what are we actually optimizing?' before writing a single line of code.

## Background

ML system design is distinct from algorithm design. You are not being asked to derive backpropagation - you are being asked to design a system that serves a business objective using ML. The interviewer wants to see systems thinking: understanding trade-offs, constraints, failure modes, and how all the pieces fit together. This notebook gives you a repeatable framework you can apply to any ML design problem.

In [ ]:
%pip install -q pandas numpy matplotlib

In [ ]:
import json
import textwrap
from dataclasses import dataclass, field
from typing import List, Dict, Optional
print("Ready.")

## 1. The Six-Stage Framework

The most important skill in ML system design is knowing what question to ask next. A structured framework ensures you don't skip critical stages. The six stages are:

1. **Problem Framing** - What are we actually optimizing? What does success look like?
2. **Data** - What data do we have? What do we need? How is it collected?
3. **Features** - What signals encode the information we need?
4. **Model** - What architecture fits the problem? What's the right complexity?
5. **Serving** - How do we get predictions to users with acceptable latency?
6. **Monitoring** - How do we know the system is working after deployment?

**Interview tip**: Always start with Problem Framing. Jumping straight to models is the #1 anti-pattern. Interviewers are explicitly watching for this.

In [ ]:
@dataclass
class MLSystemDesignTemplate:
    problem_name: str
    business_objective: str = ""
    ml_objective: str = ""
    business_metric: str = ""
    ml_metric: str = ""
    constraints: List[str] = field(default_factory=list)
    data_sources: List[str] = field(default_factory=list)
    data_volume: str = ""
    label_strategy: str = ""
    feature_groups: Dict[str, List[str]] = field(default_factory=dict)
    model_choice: str = ""
    model_rationale: str = ""
    latency_budget_ms: Optional[int] = None
    serving_architecture: str = ""
    key_metrics: List[str] = field(default_factory=list)
    
    def render(self):
        lines = [f"## ML System Design: {self.problem_name}", ""]
        sections = {
            "Problem Framing": [
                f"Business objective: {self.business_objective}",
                f"ML objective: {self.ml_objective}",
                f"Business metric: {self.business_metric}",
                f"ML metric: {self.ml_metric}",
                f"Constraints: {', '.join(self.constraints) or 'None listed'}",
            ],
            "Data": [
                f"Sources: {', '.join(self.data_sources)}",
                f"Volume: {self.data_volume}",
                f"Labels: {self.label_strategy}",
            ],
            "Model": [
                f"Choice: {self.model_choice}",
                f"Rationale: {self.model_rationale}",
            ],
            "Serving": [
                f"Latency budget: {self.latency_budget_ms}ms",
                f"Architecture: {self.serving_architecture}",
            ],
            "Monitoring": [
                f"Key metrics: {', '.join(self.key_metrics)}",
            ],
        }
        for section, items in sections.items():
            lines.append(f"### {section}")
            for item in items:
                lines.append(f"- {item}")
            lines.append("")
        return "\n".join(lines)

print("MLSystemDesignTemplate defined.")

## 2. Business Metric vs ML Metric

This is one of the most common gaps in ML system design: optimizing the ML metric does not guarantee improving the business metric. They must be aligned, but they are not the same thing.

| Business Metric | ML Metric | Gap Risk |
|---|---|---|
| Revenue per user | Click-through rate | Clicks != purchases |
| User retention (DAU) | Watch time | Watch time != satisfaction |
| Fraud loss ($) | AUC-ROC | AUC doesn't capture $ impact |
| Subscriber growth | Email open rate | Opens != subscriptions |

**Key principle**: Define the business metric first, then ask what ML metric is a reliable proxy for this, and explicitly state the assumption you're making.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 1000
true_ctr = 0.05

scores_A = np.random.beta(2, 1, n)  # overconfident model
scores_B = np.random.beta(0.5, 0.5, n)  # calibrated model
labels = (np.random.rand(n) < true_ctr).astype(int)

from scipy.stats import spearmanr
rank_corr_A = spearmanr(scores_A, labels).statistic
rank_corr_B = spearmanr(scores_B, labels).statistic
calib_error_A = abs(scores_A.mean() - labels.mean())
calib_error_B = abs(scores_B.mean() - labels.mean())

print(f"Model A - Rank correlation: {rank_corr_A:.3f}, Calibration error: {calib_error_A:.3f}")
print(f"Model B - Rank correlation: {rank_corr_B:.3f}, Calibration error: {calib_error_B:.3f}")
print()
print(f"Model A mean predicted CTR: {scores_A.mean():.3f} vs true {true_ctr:.3f}")
print(f"Model B mean predicted CTR: {scores_B.mean():.3f} vs true {true_ctr:.3f}")
print("\nConclusion: Model A wins on rank correlation but overestimates CTR by 10x.")
print("In a live auction, Model A overbids -> inflated CPCs -> advertiser ROI collapse.")

## 3. The Four Failure Modes

Most ML projects fail for one of four reasons:

1. **Wrong problem**: The ML objective doesn't align with the business objective.
2. **Wrong data**: Training on a distribution that doesn't match deployment.
3. **Wrong metric**: Optimizing a proxy that diverges from real value.
4. **Wrong deployment**: The model is accurate but too slow, or the feature pipeline doesn't match training.

**Interview framing**: After you present a design, proactively say 'Let me walk through the failure modes for this system.' This signals maturity.

In [ ]:
failure_modes = {
    "Wrong Problem": [
        "Does the ML objective directly cause the business outcome?",
        "Have you validated with a simple rule-based baseline?",
        "Is there a simpler non-ML solution that achieves 80% of the value?",
    ],
    "Wrong Data": [
        "Does your training distribution match the deployment distribution?",
        "Are there temporal leaks (future data in training)?",
        "Is there selection bias in how labels were collected?",
    ],
    "Wrong Metric": [
        "Is your offline metric correlated with your online business metric?",
        "Have you validated this correlation with past A/B test data?",
        "Is your model calibrated (predicted probs match actual frequencies)?",
    ],
    "Wrong Deployment": [
        "Does the serving feature pipeline match the training feature pipeline?",
        "Is the latency acceptable at P99, not just P50?",
        "Is there a monitoring plan to detect degradation?",
    ],
}

for mode, questions in failure_modes.items():
    print(f"[ ] {mode}")
    for q in questions:
        print(f"    - {q}")
    print()

## 4. The Data Flywheel

One of the most powerful properties of ML systems is that they can improve over time through implicit feedback loops. How it works:

1. Model makes predictions -> users interact -> interactions generate implicit labels
2. Labels improve the next model -> better model -> more useful product -> more users
3. More users -> more data -> faster flywheel

**Examples**: Google Search (clicks -> ranking labels), Spotify (saves/skips -> preferences), Gmail spam (user reports -> classifier).

**Warning**: Feedback loops can also be negative (filter bubbles, popularity bias). A good design accounts for this.

## 5. Online vs Offline Evaluation

A system that performs well offline (high AUC on held-out test set) may perform poorly online. This gap arises because:

- **Offline**: evaluates on static historical data, no feedback loop
- **Online**: users respond to the model's actual outputs in real-time
- **Position effects**: items shown higher get more clicks regardless of quality
- **Novelty effects**: new recommendations get temporary engagement boosts

**Practical rule**: Never ship a model based on offline metrics alone. Always A/B test.

In [ ]:
spam_detector = MLSystemDesignTemplate(
    problem_name="Email Spam Detector",
    business_objective="Reduce user time wasted on spam; protect user trust",
    ml_objective="Classify incoming emails as spam or not-spam",
    business_metric="Spam emails reaching inbox / total emails (miss rate)",
    ml_metric="Precision @ 99% recall",
    constraints=["<50ms latency before email delivery", "Explainability for appeals", "GDPR: no long-term content retention"],
    data_sources=["User spam reports", "User rescues from spam folder", "Known spam domains"],
    data_volume="~500M emails/day at Gmail scale",
    label_strategy="Implicit: user actions. Explicit: honeypot accounts.",
    feature_groups={
        "Content": ["TF-IDF on body text", "URL domains", "link count"],
        "Sender": ["sender domain reputation", "IP blacklist", "sending velocity"],
        "User": ["past interactions with sender", "user spam rate"],
    },
    model_choice="Logistic regression on TF-IDF + GBM on structured features -> ensemble",
    model_rationale="LR is fast, interpretable. GBM captures nonlinear sender patterns.",
    latency_budget_ms=50,
    serving_architecture="Feature lookup from Redis + real-time TF-IDF + C++ inference",
    key_metrics=["False positive rate", "False negative rate", "User appeal rate"],
)

print(spam_detector.render())

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Six-stage framework | Problem -> Data -> Features -> Model -> Serving -> Monitoring |
| Business vs ML metric | Optimizing ML metric != improving business metric |
| Four failure modes | Wrong problem, wrong data, wrong metric, wrong deployment |
| Data flywheel | Implicit feedback loops make ML systems improve over time |
| Online vs offline gap | Offline AUC does not predict online business impact; A/B test always |

### Common Pitfalls
- Jumping to model architecture before defining the ML objective
- Treating accuracy as the only metric
- Not considering the feedback loop when designing the training pipeline
- Designing for P50 latency instead of P99
- Skipping monitoring - systems degrade silently in production
